# Part 1 ↔ Part 2: complete-source dataset integration

## 1. Inspect Existing Notebooks and Data Sources

This **new, isolated notebook** leaves existing notebooks, dashboards, gold annotations and loaders unchanged. Part 1's existing acquisition function reads `nrows=50_000` from the official gzip-compressed TSV, then saves reduced CSVs. Those extracts cannot establish full-database overlap. Part 2 is `openfoodfacts/nutrient-detection-layout`, with original OCR words, boxes and BIO IDs. Both official splits are required.

The notebook imports its isolated helper from `scripts/dataset_integration_support.py`. No model weights, training, OCR inference, paid API, full export download or image payload download is needed. All generated files go under `outputs/part1_part2_integration/`; raw metadata is separately cached. **If a complete Part 1 export is missing, full overlap and integrated outputs remain blocked—not zero and not substituted with sample/search-index results.**

In [1]:
from pathlib import Path
import hashlib
import importlib
import json
import sys
import pandas as pd
from IPython.display import display, Markdown, HTML

ROOT = Path.cwd().resolve()
if not (ROOT / 'app.py').exists() and (ROOT / 'nutrition_extraction_solution/app.py').exists():
    ROOT = ROOT / 'nutrition_extraction_solution'
assert (ROOT / 'app.py').exists(), 'Run from the existing project root.'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import scripts.dataset_integration_support as integration
integration = importlib.reload(integration)
OUTPUT = ROOT / 'outputs/part1_part2_integration'
CACHE = ROOT / 'data/part1_part2_integration_cache'
OUTPUT.mkdir(parents=True, exist_ok=True)

# Snapshot existing files read-only, so accidental changes are detected later.
protected = [ROOT / 'app.py', ROOT / 'src/nutrition_extraction/acquire.py',
             *[path for path in ROOT.rglob('*.ipynb') if path.name != 'part1_part2_dataset_integration.ipynb' and '.venv' not in str(path)]]
protected_hashes = {str(path): hashlib.sha256(path.read_bytes()).hexdigest() for path in protected}
for relative in ['data/openfoodfacts_eda_enriched.csv.source.txt', 'data/openfoodfacts_sample.csv.source.txt']:
    print(relative, '\n', (ROOT / relative).read_text(encoding='utf-8'))
source_schema = json.loads((ROOT / 'data/openfoodfacts_source_schema.json').read_text(encoding='utf-8'))
print('Original source URL:', source_schema['source_url'])
print('Original source columns:', len(source_schema['columns']))
print(source_schema['columns'])
for relative in ['eda_openfoodfacts.ipynb', 'part2/part2_dataset_eda.ipynb']:
    document = json.loads((ROOT / relative).read_text(encoding='utf-8'))
    relevant = [''.join(cell.get('source', [])) for cell in document['cells'] if any(term in ''.join(cell.get('source', [])) for term in ['read_csv', '50_000', '50000', 'DatasetConfig'])]
    print('\nExisting notebook loading evidence:', relative)
    for text in relevant:
        print(text[:2500])
print('No existing file was edited. Integration helper is separate from original loaders.')

data/openfoodfacts_eda_enriched.csv.source.txt 
 source_url=https://static.openfoodfacts.org/data/en.openfoodfacts.org.products.csv.gz
rows=50000
columns=44

data/openfoodfacts_sample.csv.source.txt 
 source_url=https://static.openfoodfacts.org/data/en.openfoodfacts.org.products.csv.gz
rows=50000
columns=140

Original source URL: https://static.openfoodfacts.org/data/en.openfoodfacts.org.products.csv.gz
Original source columns: 211
['code', 'url', 'creator', 'created_t', 'created_datetime', 'last_modified_t', 'last_modified_datetime', 'last_modified_by', 'last_updated_t', 'last_updated_datetime', 'product_name', 'abbreviated_product_name', 'generic_name', 'quantity', 'packaging', 'packaging_tags', 'packaging_en', 'packaging_text', 'brands', 'brands_tags', 'brands_en', 'categories', 'categories_tags', 'categories_en', 'origins', 'origins_tags', 'origins_en', 'manufacturing_places', 'manufacturing_places_tags', 'labels', 'labels_tags', 'labels_en', 'emb_codes', 'emb_codes_tags', 'first_p

## 2. Locate and Validate the Full Part 1 Product Database

Set `FULL_PART1_PATH` only to a **complete, provenance-confirmed local OFF product export**, optionally adding search folders. Known sample paths are explicitly refused. The existing files carry `rows=50000` provenance. File size/name alone does not establish completeness. The scan later checks EOF, source stability and a minimum-row safety threshold; there is no 50,000-row cap.

**If absent:** reuse an existing full export from another drive/shared folder first. Otherwise obtain the official compressed TSV **once**, keep it compressed, and point this notebook to it. A sequential gzip chunk scan avoids a second uncompressed copy; a reusable Parquet conversion is optional if repeated full scans justify disk space. Confirm acquisition date/completeness and sufficient free disk for the disk-backed barcode index. The official source is https://static.openfoodfacts.org/data/en.openfoodfacts.org.products.csv.gz . No automatic full download is enabled. Exact-code API lookups may cheaply enrich Part 2 later, but do **not** substitute for full-snapshot row/unique counts or this requested overlap calculation.

In [15]:
FULL_PART1_PATH = ROOT / 'data/full_off/en.openfoodfacts.org.products.csv.gz'
acquisition = json.loads((ROOT / 'data/full_off/export_acquisition.json').read_text(encoding='utf-8'))
FULL_SOURCE_CONFIRMED_COMPLETE = (
    FULL_PART1_PATH.is_file()
    and FULL_PART1_PATH.stat().st_size == acquisition['expected_content_length']
    and acquisition['sampling'] is False
)
EXTRA_SEARCH_ROOTS = [Path.home() / 'Downloads']
CHUNK_ROWS = 10_000
ALLOW_PART2_METADATA_REQUESTS = False  # Both full splits are already cached.

local_inventory = integration.inventory(ROOT, EXTRA_SEARCH_ROOTS)
display(pd.DataFrame(local_inventory))
full_source = integration.validate_full_source(FULL_PART1_PATH, confirmed=FULL_SOURCE_CONFIRMED_COMPLETE)
print(json.dumps(full_source, indent=2))
print('Acquisition provenance:', acquisition)
(OUTPUT / 'source_availability.json').write_text(json.dumps({
    'part1': full_source, 'local_inventory': local_inventory,
    'scan_scope': 'project data/outputs and configured search roots; not a whole-machine search',
    'sample_fallback_allowed': False, 'full_export_download_performed': FULL_SOURCE_CONFIRMED_COMPLETE,
    'acquisition': acquisition,
}, indent=2), encoding='utf-8')
if not full_source['available']:
    display(Markdown('**BLOCKED until the complete transfer finishes.** No sample fallback is allowed.'))

,path,bytes,known_sample,provenance
0,C:\Users\SaniaJawaid\Documents\projects\practi...,27113821,True,source_url=https://static.openfoodfacts.org/da...
1,C:\Users\SaniaJawaid\Documents\projects\practi...,23295826,True,source_url=https://static.openfoodfacts.org/da...
2,C:\Users\SaniaJawaid\Documents\projects\practi...,1524849117,False,not established; a file name/size does not pro...
3,C:\Users\SaniaJawaid\Documents\projects\practi...,3304593,False,not established; a file name/size does not pro...
4,C:\Users\SaniaJawaid\Documents\projects\practi...,1075878,False,not established; a file name/size does not pro...
5,C:\Users\SaniaJawaid\Documents\projects\practi...,1662,False,not established; a file name/size does not pro...
6,C:\Users\SaniaJawaid\Documents\projects\practi...,215,False,not established; a file name/size does not pro...
7,C:\Users\SaniaJawaid\Documents\projects\practi...,907,False,not established; a file name/size does not pro...


{
  "available": true,
  "path": "C:\\Users\\SaniaJawaid\\Documents\\projects\\practice\\study\\nutrition_extraction_solution\\data\\full_off\\en.openfoodfacts.org.products.csv.gz",
  "bytes": 1524849117,
  "modified_ns": 1793340947863977100,
  "completeness": "explicit source-provenance confirmation; full EOF scan still required"
}
Acquisition provenance: {'source_url': 'https://static.openfoodfacts.org/data/en.openfoodfacts.org.products.csv.gz', 'expected_content_length': 1524849117, 'etag': 'c528d873936bb42b864ca5d146871a46-291', 'source_last_modified': 'Thu, 29 Oct 2026 09:18:24 GMT', 'source_version_id': 'vBuPnZ65RNTNxC3aUsuuqFJ.rPwdFHEU', 'format': 'gzip-compressed TSV', 'acquisition_method': 'complete HTTP transfer, resumable partial file promoted only after curl success and exact Content-Length check', 'sampling': False, 'note': 'This is acquisition intent/provenance, not a completion marker. Require final gzip file with expected byte length and successful full EOF gzip/TSV sca

## 3. Load Complete Part 1 Data Efficiently

Prepare a chunk iterator now; the **one complete EOF scan** runs after all Part 2 target barcodes are known (Section 6). This avoids scanning a huge gzip twice. CSV/TSV barcodes are read as strings with no NA inference. Parquet is scanned by record batches, and a numeric `code` column is refused because leading zeros may already have been lost. SQLite stores the full unique-barcode counts on disk with a 32 MiB page cache; only matched product payloads are retained. The source is never loaded whole into pandas.

Barcode normalization trims outer whitespace and accepts ASCII digits only. There is no padding, `.0` removal, integer conversion or fuzzy matching. Missing/invalid raw IDs remain in the image inventory and are not matched. All source rows—including duplicate codes—contribute to total row counts.

In [3]:
import shutil
import sqlite3
import pyarrow as pa
import pyarrow.parquet as pq
print({'chunk_rows': CHUNK_ROWS, 'sqlite_page_cache_MiB': 32,
       'free_disk_GiB': round(shutil.disk_usage(ROOT).free / 1024**3, 2),
       'pyarrow': pa.__version__, 'sqlite': sqlite3.sqlite_version,
       'part1_row_limit': None, 'full_source_available': full_source['available']})
assert integration.barcode(' 001234 ') == ('001234', 'valid')
assert integration.barcode('1234.0')[0] is None
assert integration.barcode('001234')[0] != integration.barcode('1234')[0]
scan = None
summary = None
products = linked_images = quality = None
print('Complete source scan is deferred to Section 6; no sample data is loaded for matching.')

{'chunk_rows': 10000, 'sqlite_page_cache_MiB': 32, 'free_disk_GiB': 264.86, 'pyarrow': '26.0.0', 'sqlite': '3.49.1', 'part1_row_limit': None, 'full_source_available': False}
Complete source scan is deferred to Section 6; no sample data is loaded for matching.


## 4. Inspect Part 2 Annotated Dataset Structure

Fetch **all** `train` and `test` metadata through the official datasets-server row API in pages of 100. This is intentionally separate from the existing 1–50-image demo loader, whose limit is unchanged. Each page contains `tokens`, `bboxes`, `ner_tags`, `image` and `meta`. `meta.barcode` is the join source; retain `image_id`, `image_url`, `ocr_url`, source row and both authoritative API split and `meta.split`.

Repository revision, exact label mapping, declared/observed split counts and each raw JSON page are cached. Partial/truncated pages or changed revision/counts abort rather than produce a partial denominator. The signed image URL is retained separately from the source image URL. Width/height are stored only if available from metadata; missing values stay null. No image is downloaded to fill them.

In [11]:
integration = importlib.reload(integration)
part2_manifest = None
part2_load_error = None
try:
    part2_manifest = integration.load_all_part2(CACHE / 'part2', allow_metadata_requests=ALLOW_PART2_METADATA_REQUESTS)
    print(json.dumps(part2_manifest, indent=2))
    print('Image inventory schema:', pq.read_schema(part2_manifest['images_path']))
    print('Gold token schema:', pq.read_schema(part2_manifest['annotations_path']))
    assert set(part2_manifest['split_rows']) == {'train', 'test'}
    assert part2_manifest['images'] == sum(part2_manifest['split_rows'].values())
    print('No image payloads downloaded:', part2_manifest['image_payloads_downloaded'] == 0)
except RuntimeError as error:
    part2_load_error = str(error)
    print('Part 2 inventory incomplete; no partial counts are certified:', part2_load_error)
    print('Raw fetched pages are cached for resume. Retry later if the service rate-limits requests.')

{
  "dataset": "openfoodfacts/nutrient-detection-layout",
  "revision": "7d3245ca596f0c8f41dc740ca0fa5987b1e0fd4d",
  "split_rows": {
    "train": 2884,
    "test": 199
  },
  "images": 3083,
  "annotations": 309190,
  "labels": [
    "O",
    "B-ENERGY_KJ_100G",
    "I-ENERGY_KJ_100G",
    "B-VITAMIN_D_SERVING",
    "I-VITAMIN_D_SERVING",
    "B-SODIUM_SERVING",
    "I-SODIUM_SERVING",
    "B-PROTEINS_SERVING",
    "I-PROTEINS_SERVING",
    "B-ADDED_SUGARS_SERVING",
    "I-ADDED_SUGARS_SERVING",
    "B-CALCIUM_SERVING",
    "I-CALCIUM_SERVING",
    "B-FAT_SERVING",
    "I-FAT_SERVING",
    "B-ENERGY_KJ_SERVING",
    "I-ENERGY_KJ_SERVING",
    "B-SUGARS_100G",
    "I-SUGARS_100G",
    "B-SATURATED_FAT_SERVING",
    "I-SATURATED_FAT_SERVING",
    "B-SERVING_SIZE",
    "I-SERVING_SIZE",
    "B-SALT_SERVING",
    "I-SALT_SERVING",
    "B-ENERGY_KCAL_SERVING",
    "I-ENERGY_KCAL_SERVING",
    "B-FAT_100G",
    "I-FAT_100G",
    "B-SUGARS_SERVING",
    "I-SUGARS_SERVING",
    "B-FIBER_SERVI

## 5. Extract Part 2 Barcodes, Images, and Annotation Inventory

One source row remains one image record even if a product has several images. `image_key = split:source_row` is a stable key within the recorded revision; it prevents duplicate/missing image IDs from creating fan-out. Original image IDs are always preserved. Annotation rows retain original token position, text, integer boxes, label ID and exact dataset label string. Inconsistent token/box/tag lengths are flagged, not truncated using `zip`; missing counterparts stay null.

The raw staging Parquet is a **Part 2 inventory only**, not an integrated or matched dataset. Report invalid/missing barcodes, duplicate image-ID groups, repeated barcode images, split discrepancies and annotation availability before any join.

In [6]:
images, part2_summary, target_codes = [], {}, set()
if part2_manifest is not None:
    images, part2_summary = integration.part2_inventory(part2_manifest)
    image_inventory = pd.DataFrame(images)
    target_codes = {row['barcode'] for row in images if row['barcode'] is not None}
    display(pd.Series(part2_summary, name='actual Part 2 inventory'))
    display(image_inventory[['barcode', 'image_id', 'dataset_split', 'image_width', 'image_height', 'annotation_status']].head(10))
    print('Image rows per barcode (largest first):')
    display(image_inventory.dropna(subset=['barcode']).groupby('barcode').size().sort_values(ascending=False).head(10))
    train_codes = set(image_inventory.loc[image_inventory['dataset_split'].eq('train'), 'barcode'].dropna())
    test_codes = set(image_inventory.loc[image_inventory['dataset_split'].eq('test'), 'barcode'].dropna())
    print('Cross-split barcode overlap (potential product leakage):', len(train_codes & test_codes))
    (CACHE / 'part2_inventory_summary.json').write_text(json.dumps({**part2_summary, 'cross_split_barcode_overlap': sorted(train_codes & test_codes)}, indent=2), encoding='utf-8')
else:
    print('Part 2 inventory not complete; counts remain unknown, not zero. Resume Section 4 before overlap.')

total_images                         3083
unique_valid_barcodes                3083
repeated_barcode_images                 0
barcodes_with_multiple_images           0
missing_or_invalid_barcodes             0
annotation_gaps                         0
split_metadata_disagreements            0
duplicate_barcode_image_id_groups       0
Name: actual Part 2 inventory, dtype: int64

,barcode,image_id,dataset_split,image_width,image_height,annotation_status
0,3831051012073,5,train,595,550,available_aligned
1,0059441001350,3,train,3043,3606,available_aligned
2,8003495103883,5,train,600,498,available_aligned
3,4100910005426,10,train,2294,1877,available_aligned
4,5000169411193,3,train,980,1200,available_aligned
5,5656754345536,5,train,571,655,available_aligned
6,4032600122086,5,train,1484,1200,available_aligned
7,231027870067000603004046,2,train,422,374,available_aligned
8,20250386,6,train,892,990,available_aligned
9,5400141058253,9,train,2722,1819,available_aligned


Image rows per barcode (largest first):


barcode
00000632         1
0000144019029    1
0002000000714    1
0002000003012    1
0003413052826    1
00051705         1
00065252         1
00074211         1
00079969         1
0008077102146    1
dtype: int64

Cross-split barcode overlap (potential product leakage): 0


## 6. Compute Barcode Overlap Against the Full Part 1 Database

Only a confirmed full source is scanned. All rows are read through EOF with no `nrows` limit, and exact unique/duplicate counts use the disk-backed full barcode index. The overlap denominator is **all unique valid Part 2 barcodes across both splits**. Report both Part 2-relative and Part 1-relative percentages, matched image rows, unmatched barcode list, total product rows and duplicate counts. No sample-based zero overlap or search-index lookup is imported as the result.

If blocked, the summary CSV records Part 1 totals and overlap as **null/not calculated**, alongside the actually observed Part 2 counts. This is a status artifact, not a fabricated statistical result.

In [16]:
scan = None  # Prevent stale success from a previous notebook run.
if full_source['available'] and part2_manifest is not None:
    scan = integration.scan_complete_part1(FULL_PART1_PATH, target_codes, CACHE / 'part1_scan',
        confirmed=FULL_SOURCE_CONFIRMED_COMPLETE, chunk_rows=CHUNK_ROWS)
    summary = integration.overlap_summary(scan, images, part2_summary)
    display(pd.Series(summary, name='actual full-source overlap'))
    print('Matched barcode examples:', scan['matched_barcodes'][:10])
    print('Unmatched barcode count:', len(scan['unmatched_barcodes']))
    display(pd.DataFrame({'unmatched_barcode': scan['unmatched_barcodes']}))
    (OUTPUT / 'full_scan_summary.json').write_text(json.dumps(scan, indent=2), encoding='utf-8')
else:
    summary = {'status': 'blocked_complete_part1_source_unavailable' if not full_source['available'] else 'blocked_part2_inventory_incomplete',
               'part1_product_rows': None, 'part1_unique_barcodes': None,
               'overlapping_barcodes': None, 'overlap_percent_of_part2': None,
               'matched_images': None, 'unmatched_barcodes': None,
               'part2_images': part2_summary.get('total_images'),
               'part2_unique_barcodes': part2_summary.get('unique_valid_barcodes')}
    display(pd.Series(summary, name='honest blocked status'))
    print('Unmatched codes cannot be classified until a complete Part 1 scan succeeds.')
summary_table = pd.DataFrame([{'metric': key, 'value': value,
    'scope': 'complete Part 1 only; null means not calculated, never sample fallback'} for key, value in summary.items()])
summary_table.to_csv(OUTPUT / 'dataset_overlap_summary.csv', index=False)
OVERLAP_CONFIRMED = scan is not None and scan.get('status') == 'complete_eof_scan'
print('Integration/export gate:', OVERLAP_CONFIRMED)

status                            measured_complete_part1_snapshot
part1_product_rows                                         4881991
part1_unique_barcodes                                      4881935
part1_duplicate_rows                                            56
part1_duplicate_barcode_groups                                  56
part2_images                                                  3083
part2_unique_barcodes                                         3083
part2_repeated_barcode_images                                    0
overlapping_barcodes                                          3075
overlap_percent_of_part2                                 99.740512
overlap_percent_of_part1                                  0.062987
matched_images                                                3075
unmatched_barcodes                                               8
invalid_part2_barcode_images                                     0
Name: actual full-source overlap, dtype: object

Matched barcode examples: ['00000632', '0000144019029', '0002000000714', '0002000003012', '0003413052826', '00051705', '00065252', '00074211', '00079969', '0008077102146']
Unmatched barcode count: 8


,unmatched_barcode
0,11213260
1,2444021034046
2,2444021039621
3,3242272251668
4,3248451011559
5,3596690765659
6,5614939243329
7,7613035253926


Integration/export gate: True


## 7. Build Linked Products, Images, and Annotations Tables

After confirmed full-source overlap, construct a **left join from all Part 2 image rows** to the matched Part 1 products. Products contains one row per valid Part 2 barcode, including unmatched placeholders—not a redundant copy of the complete product database. Images and Annotations keep `image_key`, barcode and original image ID; product text/nutrients are not repeated per token.

`match_status`: `barcode_matched_image_unverified`, `barcode_unmatched`, or `invalid_or_missing_barcode` (image-level). Invalid/missing raw IDs remain preserved on Images; no false product identity is invented. Duplicate Part 1 barcode payloads use the first source row deterministically, recording occurrence count and conflicts rather than merging different nutrient versions. Gold tokens/boxes/IDs/labels are unchanged. Final files are not exported until Section 9.

In [17]:
if OVERLAP_CONFIRMED:
    products, linked_images, quality = integration.export_linked(scan, images, part2_manifest, OUTPUT, summary, write_outputs=False)
    display(pd.DataFrame(products).head(10))
    display(pd.DataFrame(linked_images)[['image_key', 'barcode', 'image_id', 'dataset_split', 'match_status', 'image_reference_status']].head(10))
    print('Constructed in-memory small linked dimensions; gold token table remains Parquet-backed.')
else:
    print('BLOCKED: linked datasets were not constructed because full-source overlap is unconfirmed.')

,barcode,product_name,brand,category,country,serving_size,nutrition_image_url,structured_nutrients,match_status,source_record_count,source_row,conflicting_product_records
0,00000632,Risotto royal,None,None,en:fr,None,https://images.openfoodfacts.org/images/produc...,"{""carbohydrates_100g"": ""74.7"", ""energy-kcal_10...",barcode_matched_image_unverified,1,1373852.0,False
1,0000144019029,Pain au chocolat pur beurre,None,"Snacks, Snacks sucrés, Viennoiseries, Pains au...",en:fr,None,https://images.openfoodfacts.org/images/produc...,"{""carbohydrates_100g"": ""41.7"", ""energy-kcal_10...",barcode_matched_image_unverified,1,1118630.0,False
2,0002000000714,Yaourt nature brebis,La Bergerie,None,None,None,https://images.openfoodfacts.org/images/produc...,"{""added-sugars_100g"": ""0"", ""carbohydrates_100g...",barcode_matched_image_unverified,1,206.0,False
3,0002000003012,Taboulé legumes poivron,None,"Aliments et boissons à base de végétaux, Boiss...",France,None,https://images.openfoodfacts.org/images/produc...,"{""carbohydrates_100g"": ""22"", ""energy-kcal_100g...",barcode_matched_image_unverified,1,222.0,False
4,0003413052826,Gnocchi pomme de terre,Bofrost,"Aliments et boissons à base de végétaux, Alime...",France,None,https://images.openfoodfacts.org/images/produc...,"{""carbohydrates_100g"": ""40.3"", ""energy-kcal_10...",barcode_matched_image_unverified,1,274.0,False
5,00051705,2 garlic & coriander naans,Marks And Spencer,"Aliments et boissons à base de végétaux, Alime...","France, en:united-kingdom",None,https://images.openfoodfacts.org/images/produc...,"{""carbohydrates_100g"": ""48.2"", ""energy-kcal_10...",barcode_matched_image_unverified,1,1174.0,False
6,00065252,Pancake,Marks &Spencer,None,None,1 pancake (75 g),https://images.openfoodfacts.org/images/produc...,"{""added-sugars_100g"": ""2"", ""carbohydrates_100g...",barcode_matched_image_unverified,1,1201.0,False
7,00074211,Wobbler & A4 & 4in1 Price card big & small,Coffea,None,en:fr,None,https://images.openfoodfacts.org/images/produc...,"{""carbohydrates_100g"": ""29"", ""energy-kcal_100g...",barcode_matched_image_unverified,1,3343581.0,False
8,00079969,Dry Roasted & Salted Almonds,Trader Joe's,None,None,1 serving (30 g),https://images.openfoodfacts.org/images/produc...,"{""added-sugars_100g"": ""0"", ""alcohol_100g"": ""0""...",barcode_matched_image_unverified,1,1270.0,False
9,0008077102146,Hello Panda,"meiji, Meiji Seika (S) Pte. Ltd.","Snacks, Sweet snacks, Cocoa and its products, ...",United States,"4 COOKIES, PER CONTAINER ABOUT (15 g)",https://images.openfoodfacts.org/images/produc...,"{""calcium_100g"": ""0.133"", ""carbohydrates_100g""...",barcode_matched_image_unverified,1,349.0,False


,image_key,barcode,image_id,dataset_split,match_status,image_reference_status
0,train:0,3831051012073,5,train,barcode_matched_image_unverified,different_image_references
1,train:1,0059441001350,3,train,barcode_matched_image_unverified,different_image_references
2,train:2,8003495103883,5,train,barcode_matched_image_unverified,different_image_references
3,train:3,4100910005426,10,train,barcode_matched_image_unverified,different_image_references
4,train:4,5000169411193,3,train,barcode_matched_image_unverified,different_image_references
5,train:5,5656754345536,5,train,barcode_matched_image_unverified,different_image_references
6,train:6,4032600122086,5,train,barcode_matched_image_unverified,different_image_references
7,train:7,231027870067000603004046,2,train,barcode_matched_image_unverified,different_image_references
8,train:8,20250386,6,train,barcode_matched_image_unverified,different_image_references
9,train:9,5400141058253,9,train,barcode_matched_image_unverified,different_image_references


Constructed in-memory small linked dimensions; gold token table remains Parquet-backed.


## 8. Analyze Integration Quality and Match Status

Measure metadata and each structured-nutrient field's nonempty coverage **among matched products**, with explicit denominator. Show image-reference metadata comparisons, duplicates/conflicts, missing dimensions/IDs and cross-split products. URL/path/filename-ID agreement is not a verified image match. No pixels are compared, and **current product nutrients may describe different packaging from the annotated image**. Split assignments are retained; any later model split must group by barcode to avoid leakage.

Show up to 10 linked product examples (fewer only if fewer matches exist), with image hyperlinks, structured nutrients and a filtered BIO token preview. Images are hyperlinks by default: viewing them is explicit and optional; no automatic image payload download occurs.

In [18]:
import html
import pyarrow.dataset as pads
DISPLAY_REMOTE_IMAGES = False  # Explicit opt-in; False creates only clickable image links.
if OVERLAP_CONFIRMED:
    print(json.dumps(quality, indent=2))
    examples = [row for row in products if row['source_record_count']][:10]
    print('Matched product examples available:', len(examples))
    annotations_dataset = pads.dataset(part2_manifest['annotations_path'], format='parquet')
    for product in examples:
        display(Markdown(f"### Barcode {product['barcode']} — {product['product_name'] or '(name missing)'}"))
        display(pd.Series(product))
        print('Structured nutrients (source fields, raw values; not image gold):')
        print(json.dumps(json.loads(product['structured_nutrients']), ensure_ascii=False, indent=2))
        product_images = [image for image in linked_images if image['barcode'] == product['barcode']]
        for image in product_images:
            display(pd.Series({key: image[key] for key in ['image_key', 'image_id', 'dataset_split', 'image_url', 'match_status', 'image_reference_status']}))
            url = image['image_url'] or image['dataset_image_url']
            if url and url.startswith(('https://', 'http://')):
                display(HTML(f'<a href="{html.escape(url, quote=True)}" target="_blank" rel="noopener">Open annotated nutrition image (external request only when clicked)</a>'))
                if DISPLAY_REMOTE_IMAGES:
                    from IPython.display import Image as NotebookImage
                    display(NotebookImage(url=url, width=360))
            preview = annotations_dataset.scanner(filter=pads.field('image_key') == image['image_key']).head(30).to_pandas()
            display(preview)
    if not examples:
        print('Full-source overlap is measured as zero: no matched examples exist; unmatched images remain retained.')
else:
    print('NOT CALCULATED: metadata/nutrient coverage, image-reference comparisons and linked examples require the complete Part 1 source. No matched examples are fabricated.')

{
  "matched_product_count": 3075,
  "unmatched_valid_product_count": 8,
  "coverage_denominator": "unique barcode-matched products; null coverage if zero matched products",
  "metadata_coverage": {
    "product_name": 0.9928455284552845,
    "brand": 0.7183739837398374,
    "category": 0.32162601626016263,
    "country": 0.625040650406504,
    "serving_size": 0.23934959349593496,
    "nutrition_image_url": 1.0,
    "structured_nutrients": 0.9986991869918699
  },
  "nutrient_field_coverage": {
    "added-sugars_100g": 0.3921951219512195,
    "alcohol_100g": 0.15902439024390244,
    "alpha-linolenic-acid_100g": 0.03902439024390244,
    "arachidonic-acid_100g": 0.0383739837398374,
    "beta-carotene_100g": 0.13626016260162602,
    "biotin_100g": 0.000975609756097561,
    "butyric-acid_100g": 0.03902439024390244,
    "caffeine_100g": 0.006178861788617886,
    "calcium_100g": 0.17138211382113822,
    "capric-acid_100g": 0.03902439024390244,
    "caproic-acid_100g": 0.03902439024390244,
   

### Barcode 00000632 — Risotto royal

barcode                                                                 00000632
product_name                                                       Risotto royal
brand                                                                       None
category                                                                    None
country                                                                    en:fr
serving_size                                                                None
nutrition_image_url            https://images.openfoodfacts.org/images/produc...
structured_nutrients           {"carbohydrates_100g": "74.7", "energy-kcal_10...
match_status                                    barcode_matched_image_unverified
source_record_count                                                            1
source_row                                                               1373852
conflicting_product_records                                                False
dtype: object

Structured nutrients (source fields, raw values; not image gold):
{
  "carbohydrates_100g": "74.7",
  "energy-kcal_100g": "344",
  "energy_100g": "1439",
  "fat_100g": "0.8",
  "proteins_100g": "8.6",
  "saturated-fat_100g": "0.1",
  "sugars_100g": "1.1"
}


image_key                                                         train:802
image_id                                                                  2
dataset_split                                                         train
image_url                 https://images.openfoodfacts.org/images/produc...
match_status                               barcode_matched_image_unverified
image_reference_status                           different_image_references
dtype: object

,image_key,barcode,image_id,dataset_split,token_index,ocr_token,bounding_box,gold_bio_id,gold_bio_label
0,train:802,00000632,2,train,0,Risotto,"[0, 0, 251, 125]",0,O
1,train:802,00000632,2,train,1,Royal,"[278, 81, 398, 113]",0,O
2,train:802,00000632,2,train,2,-,"[420, 83, 429, 113]",0,O
3,train:802,00000632,2,train,3,Risotto,"[439, 80, 594, 127]",0,O
4,train:802,00000632,2,train,4,Royaie\n,"[609, 99, 757, 147]",0,O
5,train:802,00000632,2,train,5,(300g,"[344, 115, 454, 148]",0,O
6,train:802,00000632,2,train,6,e),"[466, 125, 504, 150]",0,O
7,train:802,00000632,2,train,7,Ingredients,"[250, 145, 402, 172]",0,O
8,train:802,00000632,2,train,8,-,"[413, 150, 429, 174]",0,O
9,train:802,00000632,2,train,9,Ingredionten,"[432, 152, 594, 178]",0,O


... (41 outputs left out)

barcode                                                            0008077102146
product_name                                                         Hello Panda
brand                                           meiji, Meiji Seika (S) Pte. Ltd.
category                       Snacks, Sweet snacks, Cocoa and its products, ...
country                                                            United States
serving_size                               4 COOKIES, PER CONTAINER ABOUT (15 g)
nutrition_image_url            https://images.openfoodfacts.org/images/produc...
structured_nutrients           {"calcium_100g": "0.133", "carbohydrates_100g"...
match_status                                    barcode_matched_image_unverified
source_record_count                                                            1
source_row                                                                   349
conflicting_product_records                                                False
dtype: object

Structured nutrients (source fields, raw values; not image gold):
{
  "calcium_100g": "0.133",
  "carbohydrates_100g": "66.67",
  "cholesterol_100g": "0.033",
  "energy-from-fat_100g": "167",
  "energy-kcal_100g": "533",
  "energy_100g": "2230",
  "fat_100g": "26.67",
  "fiber_100g": "6.7",
  "iron_100g": "0.0024",
  "proteins_100g": "6.67",
  "salt_100g": "0.9175",
  "saturated-fat_100g": "13.33",
  "sodium_100g": "0.367",
  "sugars_100g": "33.33",
  "trans-fat_100g": "0",
  "vitamin-a_100g": "0",
  "vitamin-c_100g": "0"
}


image_key                                                         train:252
image_id                                                                  3
dataset_split                                                         train
image_url                 https://images.openfoodfacts.org/images/produc...
match_status                               barcode_matched_image_unverified
image_reference_status                           different_image_references
dtype: object

,image_key,barcode,image_id,dataset_split,token_index,ocr_token,bounding_box,gold_bio_id,gold_bio_label
0,train:252,0008077102146,3,train,0,Co\n,"[0, 249, 46, 288]",0,O
1,train:252,0008077102146,3,train,1,MARU\n,"[165, 98, 217, 109]",0,O
2,train:252,0008077102146,3,train,2,ora\n,"[144, 113, 221, 130]",0,O
3,train:252,0008077102146,3,train,3,ARTIFICIAL\n,"[142, 140, 214, 148]",0,O
4,train:252,0008077102146,3,train,4,BUBBL\n,"[140, 151, 211, 164]",0,O
5,train:252,0008077102146,3,train,5,NET,"[147, 227, 180, 235]",0,O
6,train:252,0008077102146,3,train,6,WT\n,"[183, 227, 209, 235]",0,O
7,train:252,0008077102146,3,train,7,MARI\n,"[159, 255, 209, 266]",0,O
8,train:252,0008077102146,3,train,8,Stra\n,"[126, 270, 200, 286]",0,O
9,train:252,0008077102146,3,train,9,ARTIFICIA\n,"[136, 293, 205, 301]",0,O


## 9. Export Parquet Tables and Overlap Summary

Final files are exported **only after** the complete EOF scan, displayed overlap and join validation. Output root: `outputs/part1_part2_integration/`.

### Concise data dictionary

| Table | Grain / key | Fields and interpretation |
|---|---|---|
| `integrated_products.parquet` | One valid Part 2 barcode (unique) | `barcode` string; `product_name`, `brand`, `category`, `country`, `serving_size`, `nutrition_image_url` nullable strings; `structured_nutrients` JSON string with original basis-qualified source names/raw values; `match_status`; `source_record_count`, `source_row`, `conflicting_product_records`. Unmatched placeholders retain barcode, not invented metadata. |
| `integrated_images.parquet` | One original Part 2 source row; `image_key` unique | String barcode/raw barcode/status, original `image_id`, image URLs, OCR URL, authoritative `dataset_split`, `meta_split`, `source_row`, nullable pixel dimensions, annotation availability/status/counts, raw metadata JSON, `match_status`, `image_reference_status`, `image_pixels_verified=False`. A product may have several image rows. |
| `integrated_annotations.parquet` | (`image_key`, `token_index`) | Barcode, original image ID, split, OCR token, integer `[x0,y0,x1,y1]` box in original `[0,1000]` coordinates, unchanged gold ID and BIO label. Annotation image relation uses `image_key`, not barcode alone. |
| `dataset_overlap_summary.csv` | One metric per row | Actual counts/percentages and scope; blank values mean not calculated if full source is absent. |

Additional outputs include unmatched barcode list, source/revision provenance and per-field quality coverage. Assertions check no image loss/fan-out, unique product/image keys, valid annotation foreign keys, preserved token counts and unchanged annotation-file bytes. Snapshot hashes verify that original notebooks/dashboard/loaders were not modified. No integrated Parquet is produced in the blocked state.

In [19]:
if OVERLAP_CONFIRMED:
    products, linked_images, quality = integration.export_linked(scan, images, part2_manifest, OUTPUT, summary, write_outputs=True)
    annotation_source = Path(part2_manifest['annotations_path'])
    annotation_output = OUTPUT / 'integrated_annotations.parquet'
    assert hashlib.sha256(annotation_source.read_bytes()).digest() == hashlib.sha256(annotation_output.read_bytes()).digest()
    assert pq.ParquetFile(OUTPUT / 'integrated_images.parquet').metadata.num_rows == len(images)
    assert pq.ParquetFile(annotation_output).metadata.num_rows == part2_manifest['annotations']
    print('Exported validated linked tables:')
    for name in ['integrated_products.parquet', 'integrated_images.parquet', 'integrated_annotations.parquet', 'dataset_overlap_summary.csv']:
        print(name, (OUTPUT / name).stat().st_size, 'bytes')
else:
    print('BLOCKED: integrated_products.parquet, integrated_images.parquet and integrated_annotations.parquet were NOT generated. Supply a complete Part 1 export, confirm its provenance, then rerun from Section 2.')
    print('Saved only actual source availability, Part 2 staging inventory and a status CSV; Part 1 totals/overlap are unknown.')

dictionary_text = '''# Integration data dictionary\n\nProducts: unique valid Part2 barcode; nullable Part1 product_name/brand/category/country/serving_size/nutrition_image_url; structured_nutrients is JSON of original source nutrient field names and raw values. Barcode is always a string. Source duplicates are counted and conflicting payloads flagged; first source row is selected deterministically.\n\nImages: image_key=split:source_row is primary key in the recorded dataset revision. Original image_id, barcode_raw, barcode status, URLs, splits, dimensions and annotation status are preserved. Match status does not establish packaging identity. Invalid barcode images are retained without invented product metadata.\n\nAnnotations: image_key + token_index is primary key; barcode/image_id/split support audit. Original OCR token, integer bounding_box, gold_bio_id and gold_bio_label remain unchanged. No product metadata is repeated per token. Annotation arrays with missing counterparts remain null and inconsistent image rows are flagged.\n\nSummary: metric/value/scope. Unknown values stay blank while full source is unavailable. Products cover Part2 barcodes only, while full Part1 counts come from a complete disk-backed EOF scan.\n\nValidation: unique products/image keys, no Part2 image loss, annotation foreign keys and counts, annotation Parquet byte hashes, source file stability and original notebook/dashboard hashes.\n\nWarning: barcode, URL or filename ID agreement is not verified pixel/packaging identity; current nutrient metadata is not image-grounded gold. Official train/test splits and cross-split barcode risks are retained.\n'''
(OUTPUT / 'data_dictionary.md').write_text(dictionary_text, encoding='utf-8')
for path, digest in protected_hashes.items():
    assert hashlib.sha256(Path(path).read_bytes()).hexdigest() == digest, f'Protected file changed externally: {path}'
print('All protected original notebook/dashboard/loader hashes are unchanged.')

Exported validated linked tables:
integrated_products.parquet 427140 bytes
integrated_images.parquet 1077033 bytes
integrated_annotations.parquet 3304593 bytes
dataset_overlap_summary.csv 1054 bytes
All protected original notebook/dashboard/loader hashes are unchanged.


### Completed execution — full product export integration

The missing official Part 1 export was acquired once as a **1,524,849,117-byte gzip TSV**, retaining its URL, ETag, version ID and last-modified provenance. The compressed source was scanned completely through EOF in 10,000-row chunks; it was not loaded whole into RAM or replaced by the 50,000-row sample.

**Actual measured results:**

- Part 1: **4,881,991 product rows**, **4,881,935 unique string barcodes**, **56 duplicate rows across 56 barcode groups**.
- Part 2: **2,884 train + 199 test = 3,083 images**, **3,083 unique barcodes**, **309,190 unchanged gold token annotations**.
- Exact barcode overlap: **3,075 / 3,083 = 99.7405%**; eight barcodes unmatched. All unmatched images and their annotations remain in the left-joined output.
- Products and Images outputs each contain **3,083 rows**, while Annotations contains **309,190 rows**. Products represents the annotated barcode universe with nullable unmatched placeholders, not a second copy of all 4.88 million products.
- Among matched products, name coverage is **99.28%**, brand **71.84%**, category **32.16%**, country **62.50%**, serving size **23.93%**, structured nutrients **99.87%**, nutrition-image URL **100%**.
- All 3,075 matched image references differ as URL/filename metadata; **no pixel identity or packaging-version match is claimed**. Raw image URLs and dimensions are preserved. No image payloads were downloaded.

The requested Parquet tables, overlap CSV, unmatched barcode list, quality/provenance reports and data dictionary were exported under `outputs/part1_part2_integration/`. Ten real linked examples with image links, source nutrients and BIO annotations are displayed above. Original notebooks, dashboard and loader hashes remain unchanged.

Rerunning Section 6 performs a new full scan; inspect the saved summary/output files when only reading existing results. The historical zero-overlap report for the 50,000-row sample does not describe this completed full-source integration.

In [20]:
# Final read-only validation and persisted execution status; no extra validation package.
import subprocess
validation_run = subprocess.run([sys.executable, '-m', 'pytest', '-q', str(ROOT / 'tests')], cwd=ROOT, capture_output=True, text=True)
print(validation_run.stdout)
if validation_run.returncode:
    print(validation_run.stderr)
    raise RuntimeError('Regression tests failed; review before using integration outputs.')
notebook_document = json.loads((ROOT / 'part1_part2_dataset_integration.ipynb').read_text(encoding='utf-8'))
assert notebook_document['nbformat'] == 4 and isinstance(notebook_document['cells'], list)
for number, cell in enumerate(notebook_document['cells'], 1):
    assert cell['cell_type'] in {'code', 'markdown', 'raw'}
    if cell['cell_type'] == 'code':
        compile(''.join(cell['source']), f'integration_cell_{number}', 'exec')
for protected_path, original_hash in protected_hashes.items():
    assert hashlib.sha256(Path(protected_path).read_bytes()).hexdigest() == original_hash
assert len(images) == sum(part2_manifest['split_rows'].values())
assert pq.ParquetFile(part2_manifest['annotations_path']).metadata.num_rows == part2_manifest['annotations']
for field_name, filename in [('images_sha256', part2_manifest['images_path']), ('annotations_sha256', part2_manifest['annotations_path'])]:
    assert hashlib.sha256(Path(filename).read_bytes()).hexdigest() == part2_manifest[field_name]
execution_status = {
    'status': 'ready_for_complete_source' if not OVERLAP_CONFIRMED else 'integration_exported',
    'complete_part1_available': full_source['available'],
    'part2_split_rows': part2_manifest['split_rows'],
    'part2_image_rows': len(images), 'part2_unique_barcodes': part2_summary['unique_valid_barcodes'],
    'part2_annotation_rows': part2_manifest['annotations'],
    'overlap_calculated': OVERLAP_CONFIRMED,
    'integrated_parquet_exported': OVERLAP_CONFIRMED,
    'original_protected_files_unchanged': True,
    'test_output': validation_run.stdout,
    'blocking_requirement': None if OVERLAP_CONFIRMED else 'Complete provenance-confirmed local Part 1 export path required; no sample fallback.',
}
(OUTPUT / 'execution_status.json').write_text(json.dumps(execution_status, indent=2), encoding='utf-8')
print(json.dumps(execution_status, indent=2))

................................................................         [100%]
64 passed in 4.32s

{
  "status": "integration_exported",
  "complete_part1_available": true,
  "part2_split_rows": {
    "train": 2884,
    "test": 199
  },
  "part2_image_rows": 3083,
  "part2_unique_barcodes": 3083,
  "part2_annotation_rows": 309190,
  "overlap_calculated": true,
  "integrated_parquet_exported": true,
  "original_protected_files_unchanged": true,
  "test_output": "\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b\u001b.\u001b